In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import ( col, when, trim, lit, regexp_replace, split, explode, lower, array_contains, transform, isnan, count, round)
from pyspark.sql.types import IntegerType
from delta import *

warehouse_location = 'hdfs://hdfs-nn:9000/projeto/silver'

builder = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("ETL-Amazon-Credits-Silver") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("SparkSession iniciada com sucesso e Delta Lake configurado.")

SparkSession iniciada com sucesso e Delta Lake configurado.


In [2]:
hdfs_path = "hdfs://hdfs-nn:9000/projeto/bronze/amazon_credits.csv"

In [3]:
df_bronze = spark.read.csv(
    hdfs_path,
    header=True,
    inferSchema=True,
    quote='\"',
    escape='\"',
    multiLine=True
)

In [4]:
print(f"Total de registos lidos da camada Bronze: {df_bronze.count()}")
df_bronze.printSchema()
df_bronze.toPandas()

Total de registos lidos da camada Bronze: 124235
root
 |-- person_id: integer (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- character: string (nullable = true)
 |-- role: string (nullable = true)



,person_id,id,name,character,role
0,59401,ts20945,Joe Besser,Joe,ACTOR
1,31460,ts20945,Moe Howard,Moe,ACTOR
2,31461,ts20945,Larry Fine,Larry,ACTOR
3,21174,tm19248,Buster Keaton,Johnny Gray,ACTOR
4,28713,tm19248,Marion Mack,Annabelle Lee,ACTOR
...,...,...,...,...,...
124230,1938589,tm1054116,Sangam Shukla,Madhav,ACTOR
124231,1938565,tm1054116,Vijay Thakur,Sanjay Thakur,ACTOR
124232,728899,tm1054116,Vanya Wellens,Budhiya,ACTOR
124233,1938620,tm1054116,Vishwa Bhanu,Gissu,ACTOR


In [5]:
from pyspark.sql.functions import col, trim, count

print(f"Total de registos no 'df_bronze': {df_bronze.count()}")

# Procura por linhas onde 'character' é nulo ou uma string vazia (após trim)
character_vazios = df_bronze.filter(
    (col("character").isNull()) | (trim(col("character")) == "")
)

print(f"Total de 'character' nulos/vazios encontrados: {character_vazios.count()}")

# Mostra os 'role' distintos onde 'character' é nulo
character_vazios.select("role").distinct().show()

# Mostra uma amostra
character_vazios.select("id", "name", "character", "role").show(5)

Total de registos no 'df_bronze': 124235
Total de 'character' nulos/vazios encontrados: 16287
+--------+
|    role|
+--------+
|DIRECTOR|
|   ACTOR|
+--------+

+-------+--------------+---------+--------+
|     id|          name|character|    role|
+-------+--------------+---------+--------+
|tm19248|Clyde Bruckman|     null|DIRECTOR|
|tm19248| Buster Keaton|     null|DIRECTOR|
|tm82253|   Marion Gray|     null|   ACTOR|
|tm82253| William Wyler|     null|DIRECTOR|
|tm83884|  Howard Hawks|     null|DIRECTOR|
+-------+--------------+---------+--------+
only showing top 5 rows



In [6]:
from pyspark.sql.functions import lit

# Cria o novo DataFrame 'df_character' com a coluna corrigida
df_character = df_bronze.withColumn("character",
    when(
        (col("character").isNull()) | (trim(col("character")) == ""),
        lit("NA")  # Coloca a string literal "NA"
    ).otherwise(col("character")) # Mantém o valor original
)

In [7]:
character_vazios_depois = df_character.filter(
    (col("character").isNull()) | (trim(col("character")) == "")
)
num_vazias = character_vazios_depois.count()

if num_vazias == 0:
    print("✅ SUCESSO: Não foram encontrados mais 'character' nulos/vazios.")
else:
    print(f"❌ FALHA: Ainda existem {num_vazias} 'character' nulos/vazios.")
    character_vazios_depois.show()

character_na = df_character.filter(
    col("character") == "NA"
)
count_na = character_na.count()

print(f"Total de registos com character = 'NA': {count_na}")

character_na.select("role").distinct().show()

✅ SUCESSO: Não foram encontrados mais 'character' nulos/vazios.
Total de registos com character = 'NA': 16287
+--------+
|    role|
+--------+
|DIRECTOR|
|   ACTOR|
+--------+



In [11]:
from pyspark.sql.functions import lower, trim, col, translate, regexp_replace

# Colunas de texto a limpar
text_columns_to_clean = ["name", "character", "role"]

# O nosso DataFrame de trabalho
df_final_limpo = df_character

# Mapa de acentos
accent_map = {
    'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u',
    'ã': 'a', 'õ': 'o', 'â': 'a', 'ê': 'e', 'ô': 'o', 'ç': 'c',
    'ñ': 'n', 'ü': 'u', 'ë': 'e', 'ï': 'i', 'ö': 'o',
    'Á': 'A', 'É': 'E', 'Í': 'I', 'Ó': 'O', 'Ú': 'U',
    'Ã': 'A', 'Õ': 'O', 'Â': 'A', 'Ê': 'E', 'Ô': 'O', 'Ç': 'C',
    'Ñ': 'N', 'Ü': 'U', 'Ë': 'E', 'Ï': 'I', 'Ö': 'O'
}

# Converte o mapa em duas strings para a função translate()
# Esta é a forma mais eficiente de o fazer em Spark
from_chars = "".join(accent_map.keys())
to_chars = "".join(accent_map.values())


for col_name in text_columns_to_clean:

# Começa com a coluna original
    cleaned_col = col(col_name)

    # Remove espaços no início/fim (Trim)
    cleaned_col = trim(cleaned_col)

    # Remove acentos (Diacritic Folding)
    # Usamos translate() que é muito mais rápido que múltiplos regexp_replace
    cleaned_col = translate(cleaned_col, from_chars, to_chars)

    # Converte para minúsculo (Lowercase)
    cleaned_col = lower(cleaned_col)

    # Remover espaços duplos
    cleaned_col = regexp_replace(cleaned_col, " +", " ")

    # Atualiza o DataFrame com a coluna limpa
    df_final_limpo = df_final_limpo.withColumn(col_name, cleaned_col)

# Mostra o resultado
df_final_limpo.select(text_columns_to_clean).show(truncate=False)

+-------------------+-----------------------------------+-----+
|name               |character                          |role |
+-------------------+-----------------------------------+-----+
|joe besser         |joe                                |actor|
|moe howard         |moe                                |actor|
|larry fine         |larry                              |actor|
|buster keaton      |johnny gray                        |actor|
|marion mack        |annabelle lee                      |actor|
|glen cavender      |captain anderson                   |actor|
|jim farley         |general thatcher                   |actor|
|frederick vroom    |a southern general                 |actor|
|charles henry smith|annabelle's father                 |actor|
|joe keaton         |union general                      |actor|
|al st. john        |officer on horseback (uncredited)  |actor|
|frank barnes       |annabelle's brother                |actor|
|mike donlin        |union general      

In [12]:
df_final_limpo \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("hdfs://hdfs-nn:9000/projeto/silver/amazon_credits")

In [13]:
df_final_limpo.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("role") \
    .option("overwriteSchema", "true") \
    .option("path", "hdfs://hdfs-nn:9000/warehouse/projeto.db/amazon_credits") \
    .saveAsTable("projeto.amazon_credits")

In [14]:
spark.sql(
    """
    DESCRIBE HISTORY projeto.amazon_credits
    """
).show()

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      1|2025-11-17 16:14:...|  null|    null|CREATE OR REPLACE...|{isManaged -> fal...|null|    null|     null|          0|  Serializable|        false|{numFiles -> 2, n...|        null|Apache-Spark/3.4....|
|      0|2025-11-16 23:34:...|  null|    null|CREATE OR REPLACE...|{isManaged -> fal...|null|    null|     null|       null|  Serializable|        false|{numFiles -

In [15]:
spark.sql(
    """
    SELECT * FROM projeto.amazon_credits
    """
).show()

+---------+--------+--------------------+---------+--------+
|person_id|      id|                name|character|    role|
+---------+--------+--------------------+---------+--------+
|    28732| tm19248|      clyde bruckman|       na|director|
|    21174| tm19248|       buster keaton|       na|director|
|    13717| tm82253|       william wyler|       na|director|
|    23839| tm83884|        howard hawks|       na|director|
|    35387| tm56584|        nicholas ray|       na|director|
|    25637|tm160494|           john ford|       na|director|
|    20177| tm87233|         frank capra|       na|director|
|   102596| tm19424|      edgar g. ulmer|       na|director|
|   127174|tm116781|     gregory la cava|       na|director|
|   152676|tm112005|         dwain esper|       na|director|
|    55739| tm22806|       d.w. griffith|       na|director|
|    15246| tm88001|        orson welles|       na|director|
|    73331|tm141715|       rupert julian|       na|director|
|   152780|tm112415|    

In [16]:
spark.stop()